In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# 1. Load the dataset
# Adjust the path/filename to match your actual file location
df = pd.read_csv('india_air_quality_consolidated.csv')

# 2. Standardize column names (lowercase and strip whitespace)
df.columns = df.columns.str.strip().str.lower()

# 3. Ensure the date column is parsed properly
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by=['city', 'date']).reset_index(drop=True)

    # Extract time-based features if they don't already exist
    if 'month' not in df.columns:
        df['month'] = df['date'].dt.month
    if 'is_weekend' not in df.columns:
        df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)

# 4. Create lag feature (pm25_lag1) if needed by the model
if 'pm25_lag1' not in df.columns and 'pm25' in df.columns:
    df['pm25_lag1'] = df.groupby('city')['pm25'].shift(1)

# 5. Define required columns and drop nulls to produce df_clean
required_columns = [
    'city',
    'pm25',
    'pm10',
    'o3',
    'no2',
    'so2',
    'co',
    'month',
    'is_weekend',
    'pm25_lag1',
]

# Keep rows where required modeling columns are not null
df_clean = df.dropna(
    subset=[col for col in required_columns if col in df.columns]
).copy()

print(f'Cleaned dataset shape: {df_clean.shape}')
print(df_clean.head())

Cleaned dataset shape: (90118, 12)
    city      location       date  pm25 pm10   o3  no2  so2   co  month  \
1  Alwar  Moti Doongri 2018-01-21   165        37   21    7   11      1   
2  Alwar  Moti Doongri 2018-01-22   181        22   20    6   11      1   
3  Alwar  Moti Doongri 2018-01-23   208        21   11   12    4      1   
4  Alwar  Moti Doongri 2018-01-24   169        22   18    3    8      1   
5  Alwar  Moti Doongri 2018-01-25   142        28   16    4    6      1   

   is_weekend pm25_lag1  
1           1            
2           0       165  
3           0       181  
4           0       208  
5           0       169  


In [3]:
features = ['pm10', 'o3', 'no2', 'so2', 'co', 'month', 'is_weekend', 'pm25_lag1']
target = 'pm25'
all_cols = features + [target]

# 1. Clean string whitespaces and coerce non-numeric values to NaN
for col in all_cols:
    if col in df_clean.columns:
        # Convert spaces or empty strings to true NaN, then cast to float
        df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.strip(), errors='coerce')

# 2. Drop rows that have NaNs in any modeling columns
df_clean = df_clean.dropna(subset=all_cols).reset_index(drop=True)

# 3. Standardize city column to avoid case-sensitivity misses
df_clean['city'] = df_clean['city'].astype(str).str.strip().str.title()

cities = ['Delhi', 'Hyderabad', 'Bengaluru', 'Mumbai', 'Chennai', 'Kolkata']

# Run the training loop
for city in cities:
    city_data = df_clean[df_clean['city'] == city].copy()

    if len(city_data) > 100:
        X = city_data[features]
        y = city_data[target]

        split_idx = int(len(city_data) * 0.8)
        X_train = X.iloc[:split_idx]
        y_train = y.iloc[:split_idx]

        model_city = RandomForestRegressor(
            n_estimators=100, max_depth=10, random_state=42
        )
        model_city.fit(X_train, y_train)

        filename = f'rf_model_{city.lower()}.pkl'
        joblib.dump(model_city, filename)
        print(f"Model saved for {city}: {filename}")
    else:
        print(f"Not enough data for {city} (found {len(city_data)} rows)")

Model saved for Delhi: rf_model_delhi.pkl
Model saved for Hyderabad: rf_model_hyderabad.pkl
Not enough data for Bengaluru (found 0 rows)
Model saved for Mumbai: rf_model_mumbai.pkl
Model saved for Chennai: rf_model_chennai.pkl
Not enough data for Kolkata (found 0 rows)


In [4]:
print(sorted(df_clean['city'].dropna().unique()))

['Alwar', 'Chennai', 'Chikkamagaluru', 'Delhi', 'Faridabad', 'Ghaziabad', 'Greater Noida', 'Gurugram', 'Hyderabad', 'Kanpur', 'Lucknow', 'Mumbai', 'Navi Mumbai', 'Noida', 'Patna']


In [5]:
new_cities = ['Lucknow', 'Patna']

for city in new_cities:
    city_data = df_clean[df_clean['city'] == city].copy()

    if len(city_data) > 100:
        X = city_data[features]
        y = city_data[target]

        split_idx = int(len(city_data) * 0.8)
        X_train = X.iloc[:split_idx]
        y_train = y.iloc[:split_idx]

        model_city = RandomForestRegressor(
            n_estimators=100, max_depth=10, random_state=42
        )
        model_city.fit(X_train, y_train)

        filename = f'rf_model_{city.lower()}.pkl'
        joblib.dump(model_city, filename)
        print(f"Model saved for {city}: {filename}")
    else:
        print(f"Not enough data for {city} (found {len(city_data)} rows)")

Model saved for Lucknow: rf_model_lucknow.pkl
Model saved for Patna: rf_model_patna.pkl
